In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

In [3]:
df= pd.read_csv("/content/Campaign_data.csv")
df.head()

,Campaign_ID,Campaign_Goal,Duration(days),Channel_Used,Conversion_Rate,Acquisition_Cost($),ROI,Clicks,Impressions,Age_Range,conversions,Total_spend,CTR,Revenue
0,529013,Product Launch,15,Instagram,0.15,500.0,5.790000,500.0,3000.0,35-44,75.00,37500.0,0.166667,254625.00
1,275352,Market Expansion,15,Facebook,0.01,500.0,7.210000,500.0,3000.0,45-60,5.00,2500.0,0.166667,20525.00
2,692322,Product Launch,15,Instagram,0.08,500.0,0.430000,500.0,3000.0,45-60,40.00,20000.0,0.166667,28600.00
3,675757,Increase Sales,15,Pinterest,0.03,500.0,0.909824,293.0,1937.0,25-34,8.79,4395.0,0.151265,8393.67
4,535900,Market Expansion,15,Pinterest,0.13,500.0,1.422828,293.0,1937.0,45-60,38.09,19045.0,0.151265,46142.76


In [16]:
data= df.copy

In [ ]:
df.isnull().sum()

In [5]:
df.dropna(inplace=True)

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 58876 entries, 0 to 58875
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Campaign_ID          58876 non-null  int64  
 1   Campaign_Goal        58876 non-null  object 
 2   Duration(days)       58876 non-null  int64  
 3   Channel_Used         58876 non-null  object 
 4   Conversion_Rate      58876 non-null  float64
 5   Acquisition_Cost($)  58876 non-null  float64
 6   ROI                  58876 non-null  float64
 7   Clicks               58876 non-null  float64
 8   Impressions          58876 non-null  float64
 9   Age_Range            58876 non-null  object 
 10  conversions          58876 non-null  float64
 11  Total_spend          58876 non-null  float64
 12  CTR                  58876 non-null  float64
 13  Revenue              58876 non-null  float64
dtypes: float64(9), int64(2), object(3)
memory usage: 6.7+ MB


In [7]:
x = df["Campaign_Goal", "Duration(days)", "Channel_Used", "Age_Range", "Total_spend"]
y = df["ROI"]

categoricals = ["Campaign_Goal", "Channel_Used", "Age_Range"]
numericals = ["Duration(days)", "Total_spend"]

In [20]:
# preprocessing
preprocessor = ColumnTransformer(transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), categoricals),
 ("num", "passthrough", numericals)])

# model
model = RandomForestRegressor(n_estimators=200, random_state=42)

# pipeline
pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])

In [9]:
pipeline.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Campaign_Goal',
                                                   'Channel_Used',
                                                   'Age_Range']),
                                                 ('num', 'passthrough',
                                                  ['Duration(days)',
                                                   'Total_spend'])])),
                ('model',
                 RandomForestRegressor(n_estimators=200, random_state=42))])

In [10]:
def recommend_campaign_vectorized(budget, goal, df=df, model=pipeline):
    channels = df["Channel_Used"].unique()
    durations = df["Duration(days)"].unique()
    age_ranges = df["Age_Range"].unique()

    all_combinations = pd.DataFrame(
        [(goal, dur, ch, age, budget)
         for ch in channels
         for dur in durations
         for age in age_ranges],
        columns=["Campaign_Goal", "Duration(days)", "Channel_Used", "Age_Range", "Total_spend"])

    all_combinations["Predicted_ROI"] = model.predict(all_combinations)
    best = all_combinations.loc[all_combinations["Predicted_ROI"].idxmax()]
    return best


In [22]:
best_campaign = recommend_campaign_vectorized(budget=5500, goal="Increase Sales")
print(best_campaign)

Campaign_Goal     Increase Sales
Duration(days)                15
Channel_Used             Twitter
Age_Range                  45-60
Total_spend                 5500
Predicted_ROI           5.706186
Name: 16, dtype: object
